# Example CC Path Generation

Example workflow for generating the input structures for a carrier capture calculation: symmetry-matching the initial and final defect structures, then interpolating along the configuration coordinate path.

Contributed by Seán Kavanagh ([@kavanase](https://github.com/kavanase)) — see [issue #20](https://github.com/WMD-group/CarrierCapture.jl/issues/20).

**Requirements:** [`nonrad`](https://nonrad.readthedocs.io) and [`pymatgen`](https://pymatgen.org) (Python). The convenience functions in [`doped.utils.configurations`](https://doped.readthedocs.io/en/latest/doped.utils.html#module-doped.utils.configurations) can also be used for this — see the `doped` [NEB/CCD generation tutorial](https://doped.readthedocs.io/en/latest/CCD_NEB_tutorial.html).

**Note:** the file paths below (e.g. `../VASP_Files/Int_Te_3_1/...`, a tellurium interstitial in CdTe) are placeholders — replace them with the relaxed initial and final structures of your own defect calculations.


In [ ]:
from nonrad import ccd
from pymatgen.core.structure import Structure
import numpy as np

Load your initial and final structures:

In [ ]:
i_struc = Structure.from_file("../VASP_Files/Int_Te_3_1/vasp_ncl/POSCAR")
f_struc = Structure.from_file("../VASP_Files/Int_Te_3_unperturbed_0/vasp_ncl/POSCAR")

We need to make sure that the atomic indices of the structures are symmetry matched, such that we choose the shortest linear path in interpolating between the two structures. For instance, if both structures involve a Te atom displaced away from the defect, that this is the same Te atom (index) in both structures, rather than a different Te neighbour, which would then give a different migration path.

In [ ]:
ccd.get_dQ?

In [ ]:
print(ccd.get_dQ(i_struc, f_struc)) # delta Q is the mass-weighted displacement between structures (amu^{1/2} Angstrom)
# so should typically be small (<5) for light atoms!

In [ ]:
# ensure our atomic indexing in the structures are correctly matched

from pymatgen.analysis.structure_matcher import StructureMatcher, Structure

sm = StructureMatcher(ltol=0.0001, stol=0.2, primitive_cell=False)
print(sm.fit(i_struc, f_struc))

i_like_f = sm.get_s2_like_s1(f_struc, i_struc)

# sometimes this get_s2_like_s1 doesn't work properly due to different (but equivalent) lattice vectors 
# (e.g. a=(010) instead of (100) etc.), so do this to be sure:
i_really_like_f = Structure(f_struc.lattice, f_struc.species, i_like_f.frac_coords)

In [ ]:
print(ccd.get_dQ(i_struc, f_struc))
print(ccd.get_dQ(i_really_like_f, i_struc))
print(ccd.get_dQ(i_really_like_f, f_struc)) 
# we see that this rearranges the structure so the atom indices should now match correctly. This should give a lower dQ as we
# see here (or the same if the original structures matched perfectly)

Compare the local environments of the defects, do they now correctly match?

In [ ]:
def print_neighbour_info(struc, frac_coords, cutoff_radius=3.2):
    np.set_printoptions(precision=3) # pretty print numpy array
    for neighbour in struc.get_sites_in_sphere(np.array(frac_coords)*struc.lattice.a, cutoff_radius):
        print(neighbour.species, neighbour.frac_coords, neighbour.index, 
              f"{neighbour.distance_from_point(np.array(frac_coords)*struc.lattice.a):.2f}")
    np.set_printoptions(precision=8) # change back to default after

In [ ]:
!head ../VASP_Files/Int_Te_3_1/vasp_gam/POSCAR

In [ ]:
print_neighbour_info(i_struc, [0.75, 0.25, 0.75]) # 0.75, 0.25, 0.75 is initial Te_i coords here

In [ ]:
print_neighbour_info(i_really_like_f, [0.75, 0.25, 0.75]) # 0.75, 0.25, 0.75 is initial Te_i coords here

In [ ]:
print_neighbour_info(f_struc, [0.75, 0.25, 0.75]) # 0.75, 0.25, 0.75 is initial Te_i coords here

Note that the site indices of Te closest and further away from this point now match between `i_really_like_f` and `f_struc`, whereas they didn't for `i_struc`!

Generate our interpolated structures for carrier capture calculation:

In [ ]:
import os
from pathlib import Path
from shutil import copyfile
from pymatgen.core.structure import Structure
from nonrad.ccd import get_cc_structures
import numpy as np

# output directory that will contain the input files for the CC diagram
cc_dir = "Example"
os.makedirs(cc_dir + "/disp_dir_i", exist_ok=True)
os.makedirs(cc_dir + "/disp_dir_f", exist_ok=True)

# displacement coordinates: (may need to extend for a better fit in certain cases)
displacements = np.linspace(-0.4, 0.4, 9)
displacements = np.append(np.array([-1.2, -1.0, -0.8, -0.6]), displacements)
displacements = np.append(displacements, np.array([0.6, 0.8, 1.0, 1.2]))
disp_i = i_really_like_f.interpolate(f_struc, displacements, 
                            interpolate_lattices=True,
                           pbc=True, autosort_tol=1.2)
disp_f = f_struc.interpolate(i_really_like_f, displacements, 
                            interpolate_lattices=True,
                           pbc=True, autosort_tol=1.2)

for disps in [(disp_i, "disp_dir_i"), (disp_f, "disp_dir_f")]:
    for i, struct in enumerate(disps[0]):
        string = f"{displacements[i]:.1f}".replace(".", "")
        if len(string) == 2:
            string = "0" + string
        working_dir = cc_dir + "/" + disps[1] + f"/disp_{string}"
        print(working_dir)
        os.makedirs(str(working_dir), exist_ok=True)
    
        # write structure:
        struct.to(filename = working_dir + '/POSCAR', fmt='poscar')